# Rugby Rulebook Scrum-Law Tree Extraction ("final scrum branches")

For each rulebook year, this notebook parses just the **scrum law** section
(the exact heading pattern used to find it, and how many levels of
sub-points it has, differs by year-range, same as the other tree-extraction
notebooks) into a nested YAML tree, saves it, then reloads every saved tree
to compare structure across years (a grid of all trees, and the number of
"terminating" / leaf nodes vs. year and vs. cumulative test matches).

## Requirements

```bash
pip install networkx pyyaml matplotlib numpy
```
`pygraphviz` requires Graphviz to be installed on your system (e.g.
`brew install graphviz` on macOS, `apt install graphviz libgraphviz-dev` on
Debian/Ubuntu) *before* `pip install pygraphviz`.

## Data

This notebook expects a `data/` folder (next to the notebook) laid out as:

```
data/
├── rugby_test_matches_per_year_backfilled.csv
├── rugby_laws_per_year.csv
├── hierarchical_trees/
│   ├── final_scrum_branches/   (generated by this notebook)
│   ├── flattened/
│   └── original/
├── interdependencies/
└── text_files/
    └── cleaned_relavent_texts/
        ├── 1845/
        ├── ...
        └── 2018-2026/
```


## Imports

In [ ]:
import networkx as nx
from pathlib import Path 
import yaml
import matplotlib.pyplot as plt
from networkx.drawing.nx_agraph import graphviz_layout
import re
import string
import numpy as np
import copy
import matplotlib.cm as cm
import csv

## Configuration

Update `DATA_DIR` if your data lives somewhere other than next to this
notebook.

In [ ]:
DATA_DIR = Path('data')

MATCHES_CSV_PATH = DATA_DIR / 'rugby_test_matches_per_year_backfilled.csv'
TEXTS_DIR = DATA_DIR / 'text_files' / 'cleaned_relavent_texts'
TREES_BASE = DATA_DIR / 'hierarchical_trees'
SCRUM_DIR = TREES_BASE / 'final_scrum_branches'

## Core helper functions

`create_heirarchical_tree` walks a parsed YAML tree up to 5 levels deep and
turns it into a `networkx.DiGraph`, with node color/size set by depth.
`generate_graph` then lays that graph out (`graphviz_layout`, `prog='dot'`)
and draws it, auto-scaling the axes so the tree isn't stretched. Both
functions are reused for every year-range below.

In [ ]:
def create_heirarchical_tree(data):
    colors = cm.viridis(np.linspace(0, 0.85, 10))
    
    G = nx.DiGraph()
    G.add_node('center', color=colors[0], node_size=300)
    level1_no = 0

    # adds level1 nodes, connecting to center
    for edge in data['(full ruleset)']:
        temp_data1 = data['(full ruleset)']

        level1_node = next(iter(edge))
        level1_node_name = str(level1_node + ' ')
        level1_no += 1
        level2_no = 0

        G.add_node(level1_node_name, color=colors[1], node_size=200)
        G.add_edge('center', level1_node_name)

        #print("LAW:", law_node_name)


        # adds level2 nodes, connecting to level1
        for edge in temp_data1[level1_no-1][level1_node]:
            temp_data2 = temp_data1[level1_no-1][level1_node]

            level2_node = next(iter(edge))
            level2_node_name = str(level1_node_name + level2_node + ' ')
            level2_no += 1
            level3_no = 0

            G.add_node(level2_node_name, color=colors[2], node_size=160)
            G.add_edge(level1_node_name, level2_node_name)

            #print("HEADING:", heading_node_name)


            # adds level3, connecting to level2
            for edge in temp_data2[level2_no-1][level2_node]:
                temp_data3 = temp_data2[level2_no-1][level2_node]

                level3_node = next(iter(edge))
                level3_node_name = str(level2_node_name + level3_node + ' ')
                level3_no += 1
                level4_no = 0

                G.add_node(level3_node_name, color=colors[3], node_size=120)
                G.add_edge(level2_node_name, level3_node_name)

                #print("SUBLAW:", sublaw_node_name)


                # adds level4, connecting to level3
                for edge in temp_data3[level3_no-1][level3_node]:

                    level4_node = next(iter(edge))
                    level4_node_name = str(level3_node_name + level4_node+ ' ')
                    level4_no += 1
                    level5_no = 0

                    G.add_node(level4_node_name, color=colors[4], node_size=80)
                    G.add_edge(level3_node_name, level4_node_name)

                    #print("LOWER:", subsublaw_node_name)

                    # adds level5, connecting to level4
                    for edge in temp_data3[level3_no-1][level3_node][level4_no-1][level4_node]:
                        level5_node = next(iter(edge))
                        level5_node_name = str(level4_node_name + level5_node + ' ')
                        level5_no += 1
                        level6_no = 0

                        G.add_node(level5_node_name, color=colors[5], node_size=60)
                        G.add_edge(level4_node_name, level5_node_name)

                        #print("ROMAN:", subsubsublaw_node_name)

                        # adds level6, connecting to level5
                        for edge in temp_data3[level3_no-1][level3_node][level4_no-1][level4_node][level5_no-1][level5_node]:
                            level6_node = next(iter(edge))
                            level6_node_name = str(level5_node_name + level6_node + ' ')
                            level6_no += 1
                            level7_no = 0

                            G.add_node(level6_node_name, color=colors[6], node_size=40)
                            G.add_edge(level5_node_name, level6_node_name)


                            # adds level7, connecting to level6
                            for edge in temp_data3[level3_no-1][level3_node][level4_no-1][level4_node][level5_no-1][level5_node][level6_no-1][level6_node]:
                                level7_node = next(iter(edge))
                                level7_node_name = str(level6_node_name + level7_node + ' ')
                                level7_no += 1
                                level8_no = 0

                                G.add_node(level7_node_name, color=colors[7], node_size=35)
                                G.add_edge(level6_node_name, level7_node_name)

                                # adds level8, connecting to level7
                                for edge in temp_data3[level3_no-1][level3_node][level4_no-1][level4_node][level5_no-1][level5_node][level6_no-1][level6_node][level7_no-1][level7_node]:
                                    level8_node = next(iter(edge))
                                    level8_node_name = str(level7_node_name + level8_node + ' ')
                                    level8_no += 1
                                    level9_no = 0

                                    G.add_node(level8_node_name, color=colors[8], node_size=30)
                                    G.add_edge(level7_node_name, level8_node_name)

                                    # adds level9, connecting to level8
                                    for edge in temp_data3[level3_no-1][level3_node][level4_no-1][level4_node][level5_no-1][level5_node][level6_no-1][level6_node][level7_no-1][level7_node][level8_no-1][level8_node]:
                                        level9_node = str(edge)
                                        level9_node_name = str(level8_node_name + level9_node + ' ')
                                        level9_no += 1
                                        level10_no = 0

                                        G.add_node(level9_node_name, color=colors[9], node_size=25)
                                        G.add_edge(level8_node_name, level9_node_name)



    return G
        


In [ ]:
def generate_graph(G, year):
    pos = graphviz_layout(G, prog='dot', args='-Gnodesep=0.4 -Granksep=75')
    node_colors = [G.nodes[node]['color'] for node in G.nodes()]
    node_sizes = [G.nodes[node]['node_size'] for node in G.nodes()]


    fig, ax = plt.subplots(figsize=(16,16))
    #ax.set_aspect('scaled')
    nx.draw(G, pos, ax=ax, node_size=node_sizes, node_color=node_colors, width=1, edge_color='dimgrey', arrows=False)

    # make size of axes so doesn't pull diagram
    xs = [p[0] for p in pos.values()]
    ys = [p[1] for p in pos.values()]
    max_range = max(max(xs) - min(xs), max(ys) - min(ys))
    center_x = (min(xs) + max(xs)) / 2
    center_y = (min(ys) + max(ys)) / 2
    padding = 0.05 * max_range
    half = (max_range / 2) + padding
    ax.set_xlim(center_x - half, center_x + half)
    ax.set_ylim(center_y - half, center_y + half)
    ax.set_aspect("equal")

    plt.title(f'{year}', size=40)
    #plt.annotate(f'{year}',xy=(0, 1), xytext=(10, -10), xycoords='axes fraction', textcoords='offset points', ha='left', va='top', size=40)

## 2018-2026

Parse the scrum law and save a YAML tree for every file in this range.

In [ ]:
# law 19
def create_yamls_2018_2026(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    law_no = 1
    real_law_no = 0
    level2_no = 0
    level3_no = 0
    level4_no = 0
    level5_no = 0
    in_alphabet = False
    in_definitions = True
    in_laws = False

    level0_no = 1


    for line in file:
        temp_text = line

        # main ruleset section
        if re.search(r'^(LAW|Law)\s(19)(\.|\s|:)', temp_text):
            level0_no = 1
            level0_string = str('(main ruleset)')
            yaml_data['(full ruleset)'].append({level0_string: []})
            in_laws = True
            in_definitions = False
            level2_no = 0
        if re.search(r'^(LAW|Law)\s(20)(\.|\s|:)', temp_text):
            in_laws = False
            break
    
        # return arabic numbers prior to any subheadings --> level 2
        if re.search(r'([0-9]|[0-9][0-9])\.\s', temp_text) and law_no>=1 and not in_definitions:
            in_alphabet = False
            penalty3 = False
            penalty4 = False
            level2_no += 1
            level2_string = str('sublaw ' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            level3_no = 0
            level4_no = 0
            paragraph_no = 0

        # adds lowercase letters (subsublaws) --> level 3
        subsublaw_letters = list(string.ascii_letters)
        if re.search(r'^[a-z]\.\s', temp_text) and law_no>=1 and level2_no>=1 and not in_definitions:
            paragraph_no = 0
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'^i\.\s', temp_text):
                in_alphabet = True
                level3_no += 1
                level3_string = subsublaw_letters[level3_no-1]
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
                level4_no = 0
                level5_no = 0
            elif re.match(r'^i\.\s', temp_text) and {'h': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string]:
                in_alphabet = True
                level3_no += 1
                level3_string = subsublaw_letters[level3_no-1]
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
                level4_no = 0  
                level5_no = 0

        # returns sanctions 
        if (re.search(r'^(Sanction:)\s', temp_text) or re.search(r'\s(Sanction:)\s', temp_text)) and level2_no>=1 and not in_definitions:
            paragraph_no = 0
            # same level as lowercase --> level 3
            if not in_alphabet:
                penalty3 = True
                level3_no += 1
                level3_string = str('penalty ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                level4_no = 0
                level5_no = 0
            # sublevel of lowercase --> level 4
            elif in_alphabet:
                penalty4 = True
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0


        # adds lowercase roman numerals (subsubsublaws)
        subsubsublaw_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ']
        # only adds i. if there is a ii., so doesn't add lowercase letter i. 
        if (re.search(r'^ii\.\s', temp_text) or re.search(r'\sii\.\s', temp_text)) and law_no >=1 and level2_no>=1 and level3_no>=1 and not in_definitions:
            paragraph_no = 0
            if (not penalty3 or (penalty3 and not re.search(r'^ii\.\s', temp_text))) and (not penalty4 or (penalty4 and not re.search(r'^ii\.\s', temp_text))):
                penalty3 = False
                penalty4 = False
                level4_no += 2
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({'i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({'ii. ': []})
                level5_no = 0
            elif penalty3:
                level3_no += 1
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({'penalty ii. ': []})
                level4_no = 0
                level5_no = 0
            elif penalty4:
                level4_no += 1
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({'penalty ii. ': []})
                level5_no = 0
        if (re.search(r'(iii|iv|v)\.\s', temp_text)) and law_no >=1 and level2_no>=1 and level3_no>=1 and not in_definitions:
            paragraph_no = 0
            if not penalty3 and not penalty4:
                level4_no += 1
                level4_string = subsubsublaw_labels[level4_no-1]
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
            elif penalty3:
                level3_no += 1
                level3_string = str('penalty ' + str(subsubsublaw_labels[level3_no-1]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
                level4_no = 0
                level5_no = 0
            elif penalty4:
                level4_no += 1
                level4_string = str('penalty ' + str(subsubsublaw_labels[level4_no-1]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                level5_no = 0

        # adds paragraphs
        if (re.search(r'^(?!(LAW|PRINCIPLE|Sanction|Excep|o|•|\*|-|[0-9]|\(|\s|i|v|x|DEF|[a-z]\.\s|[A-Z][A-Z])).*', temp_text)) and level2_no>=1 and real_law_no>=1 and not in_definitions:
            if law_no==0 and level3_no==0 and level4_no==0:
                paragraph_no += 1
                if paragraph_no>=2:
                    law_no += 1
                    law_string = str('P' + str(paragraph_no))
                    yaml_data['(full ruleset)'].append({law_string: []})
            elif law_no!=0 and level3_no==0 and level4_no==0:
                paragraph_no += 1
                if paragraph_no>=1:
                    level2_no += 1
                    level2_string = str('P' + str(paragraph_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            elif law_no!=0 and level3_no!=0 and level4_no==0:
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            elif law_no!=0 and level3_no!=0 and level4_no!=0:
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            

    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



Run it.

In [ ]:
initial_path_2018_2026 = (TEXTS_DIR / '2018-2026')
final_path_2018_2026 = (SCRUM_DIR / 'clean_')

for file_path in initial_path_2018_2026.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
        
        create_yamls_2018_2026(file, final_path_2018_2026, year)
        

## 2003-2016

Parse the scrum law and save a YAML tree for every file in this range.

In [ ]:
def create_yamls_2003_2016(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    level0_no = 1
    real_law_no = 1
    level2_no = 1
    level3_no = 0
    level4_no = 0
    level5_no = 0
    level6_no = 0
    in_definitions = True
    in_laws = False
    paragraph_no = 0



    for line in file:
        temp_text = line


        # main ruleset section
        if re.search(r'^(LAW|Law)\s(20)(\.|\s)', temp_text):
            level0_no = 1
            level0_string = str('(main ruleset)')
            in_definitions = False
            level2_no = 1
            in_laws = True
            yaml_data['(full ruleset)'].append({level0_string: []})
        if re.search(r'^(LAW|Law)\s(21)(\.|\s)', temp_text):
            in_laws = False
            break


        # returns defintions or sublaws -- letter sections NOT present --> level 3
        if (re.search(fr'^{real_law_no}\.([0-9]|[0-9][0-9])\s', temp_text) or re.search(fr'^{real_law_no}\.(A|B|C|D|E)\.([0-9]|[0-9][0-9])\s', temp_text) or re.search(fr'^(DEFINITION|Definition)', temp_text)) and level0_no==1 and level2_no>=1 and not in_definitions:
            in_alphabet = False
            level3_no += 1
            level3_string = str('sublaw ' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string: []})
            level4_no = 0
            level5_no = 0
            level6_no = 0
            paragraph_no = 0

        # returns lowercase letters -- letter sections NOT present --> level 4
        if re.search(r'\([a-r]\)', temp_text) and level2_no>=1 and level3_no>=1 and not in_definitions:
            in_alphabet = True
            paragraph_no = 0
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                level4_no += 1
                level4_string = str('subsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
                level5_no = 0
                level6_no = 0

            elif re.match(r'\(i\)\s', temp_text) and {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string]:
                level4_no += 1
                level4_string = str('subsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
                level5_no = 0
                level6_no = 0

        # returns bullets -- letter sections NOT present 
        if re.search(r'^(•|\*|o)\s', temp_text) and level2_no>=1 and level3_no>=1 and not in_definitions:
            # bullets BEFORE lowercase letters --> level 4
            if level4_no==0:
                paragraph_no = 0
                level4_no += 1
                level4_string = str('subsubsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
            # bullets AFTER lowercase letters --> level 5
            elif level4_no>=1:
                paragraph_no = 0
                level5_no += 1
                level5_string = str('subsubsublaw ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0

        # returns roman numerals -- letter sections NOT present --> level 5
        subsubsublaw_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ']
        # only adds i. if there is a ii., so doesn't add lowercase letter i. 
        if (re.search(r'^(ii\.|\(ii\))\s', temp_text) or re.search(r'\sii\.\s', temp_text)) and level2_no>=1 and level3_no>=1 and level4_no>=1 and not in_definitions:
            paragraph_no = 0
            level5_no += 2
            level5_string = subsubsublaw_labels[level5_no-1]
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'i. ':[]})
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
            level6_no = 0
        if (re.search(r'(iii|iv|v|\(iii|\(iv|\(v)\.\s', temp_text)) and level2_no>=1 and level3_no>=1 and level4_no>=1:
            paragraph_no = 0
            level5_no_no += 1
            subsubsublaw_string = subsubsublaw_labels[level5_no-1]
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({subsubsublaw_string:[]}) 
            level6_no = 0

        # returns sanctions -- letter sections NOT present
        if (re.search(r'^(Sanction:|Penalty:)\s', temp_text) or re.search(r'\s(Sanction:|Penalty:)\s', temp_text) or re.search(r'^(Exception:|Exception\s[0-9]:|EXCEPTION)', temp_text)) and level2_no>=1 and level3_no>=1 and not in_definitions:
            paragraph_no = 0
            # same level as lowercase --> level 4
            if not in_alphabet:
                level4_no += 1
                level4_string = str('penalty/exception ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
            # sublevel of lowercase --> level 5
            elif in_alphabet:
                level5_no += 1
                level5_string = str('penalty/exception ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0

        # adds paragraphs
        if (re.search(r'^(?!(LAW|Penalt|Sanction|Excep|Definit|o|•|\*|-|[0-9]|\(|[a-z]\)|\s|i|v|x|DEF)).*', temp_text)) and level2_no>=1 and real_law_no>=1 and not in_definitions:
            #print(temp_text)
            if level3_no==0 and level4_no==0 and level5_no==0 and level6_no==0:
                paragraph_no += 1
                if paragraph_no >=2:
                    level0_no += 1
                    level0_string = str('P' + str(paragraph_no))
                    yaml_data['(full ruleset)'].append({level0_string: []})
            if level3_no!=0 and level4_no==0 and level5_no==0 and level6_no==0:
                paragraph_no += 1
                if paragraph_no >=2:
                    level3_no += 1
                    level3_string = str('P' + str(paragraph_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no==0 and level6_no==0:
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no==0:
                paragraph_no += 1
                level6_no += 1
                level6_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})


    
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



Run it.

In [ ]:
initial_path_2003_2016 = (TEXTS_DIR / '2003-2016')
final_path_2003_2016 = (SCRUM_DIR / 'clean_')

for file_path in initial_path_2003_2016.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
        year = year.replace('2003-2016', '')
        
        create_yamls_2003_2016(file, final_path_2003_2016, year)

## 2000-2002

Parse the scrum law and save a YAML tree for every file in this range.

In [ ]:
def create_yamls_2000_2002(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 1
    level2_no = 1
    level3_no = 1
    level4_no = 0
    level5_no = 0
    level6_no = 0
    level7_no = 0
    in_definitions = False
    in_directives = False
    in_laws = False



    for line in file:
        temp_text = line

    
        if re.search(r'^(BEFORE THE MATCH)', temp_text):
            in_definitions = False


        # returns laws --> level 2
        if (re.search(r'^(LAW)\s(20)', temp_text)) and not in_definitions:
            in_laws = True
            in_definitions = False
            paragraph_no = 0
            level0_no = 1
            level0_string = str('(main ruleset)')
            yaml_data['(full ruleset)'].append({level0_string: []})
            level4_no = 0
        if (re.search(r'^(LAW)\s(21)', temp_text)) and not in_definitions:
            in_laws = False
            break

        
        # returns defintions or sublaws -- letter sections NOT present --> level 4
        if (re.search(fr'^([0-9]|[0-9][0-9])\s[A-Z]', temp_text) or re.search(fr'^([0-9]|[0-9][0-9])\.\s[A-Z]', temp_text) or (re.search(fr'^(DEFINITION)', temp_text) and level4_no==0)) and level3_no>=1 and not in_definitions and in_laws:
            paragraph_no = 0
            in_alphabet = False
            level4_no += 1
            level4_string = str('sublaw ' + str(level4_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level4_string: []})
            level5_no = 0

        # returns lowercase letters -- letter sections NOT present --> level 5
        if (re.search(r'\([a-t]\)', temp_text)) and level3_no>=1 and level4_no>=1 and not in_definitions and in_laws:
            paragraph_no = 1
            in_alphabet = True
            level5_no += 1
            level5_string = str('subsublaw ' + str(level5_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level4_no-1][level4_string].append({level5_string: []})
            level6_no = 0

        # returns bullets -- letter sections NOT present 
        if re.search(r'^-\s', temp_text) and level3_no>=1 and level4_no>=1 and not in_definitions and in_laws:
            paragraph_no = 1
            # bullets BEFORE lowercase letters --> level 5
            if level5_no==0:
                level5_no += 1
                level5_string = str('subsubsublaw ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level4_no-1][level4_string].append({level5_string:[]})
            # bullets AFTER lowercase letters --> level 6
            elif level5_no>=1:
                level6_no += 1
                level6_string = str('subsubsublaw ' + str(level6_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})

        # returns sanctions -- letter sections NOT present
        if (re.search(r'^(Penalty:)\s', temp_text) or re.search(r'\s(Penalty:)\s', temp_text) or re.search(r'^(Exception:|Exception\s[0-9]:|EXCEPTION)', temp_text)) and level3_no>=1 and not in_definitions and in_laws:
            paragraph_no = 1
            # same level as lowercase --> level 5
            if not in_alphabet:
                level5_no += 1
                level5_string = str('penalty/exception ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
            # sublevel of lowercase --> level 6
            elif in_alphabet:
                level6_no += 1
                level6_string = str('penalty/exception ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0

        
        # adds paragraphs
        if (re.search(r'^(?!(LAW|Penalt|o|•|\*|-|[0-9]|\(|\s|NOTE|DEF|(P|F)\.K\.|[A-Z][A-Z]|[A-E]\.)).*', temp_text)) and level3_no>=1 and real_law_no>=1 and not in_definitions and in_laws:
            if level4_no==0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph_no += 1
                if paragraph_no>=2:
                    level0_no += 1
                    level0_string = str('P' + str(paragraph_no))
                    yaml_data['(full ruleset)'].append({level0_string: []})
            elif level4_no!=0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph_no += 1
                if paragraph_no>=2:
                    level4_no += 1
                    level4_string = str('P' + str(paragraph_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level4_string: []})
            elif level4_no!=0 and level5_no!=0 and level6_no!=0 and level7_no==0:
                paragraph_no += 1
                level6_no += 1
                level6_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
            elif level4_no!=0 and level5_no!=0 and level6_no!=0 and level7_no!=0:
                paragraph_no += 1
                level7_no += 1
                level7_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string: []})


       
    
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



Run it.

In [ ]:
initial_path_2000_2002 = (TEXTS_DIR / '2000-2002')
final_path_2000_2002 = (SCRUM_DIR / 'clean_')

for file_path in initial_path_2000_2002.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
        year = year.replace('2000-2002', '')
        
        create_yamls_2000_2002(file, final_path_2000_2002, year)
        

## 1997-1998

Parse the scrum law and save a YAML tree for every file in this range.

In [ ]:
# adding rulings to notes
def create_yamls_1997_1998(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 1
    level2_no = 1
    level3_no = 0
    level4_no = 0
    level5_no = 0
    level6_no = 0
    level7_no = 0
    penalty4 = False
    penalty5 = False
    penalty6 = False
    notes = False
    in_definitions = False
    in_laws = False
    letter_sections = False


    for line in file:
        temp_text = line

        # returns laws --> level 2
        if (re.search(r'^(LAW)\s(20)', temp_text)):
            in_laws = True
            in_definitions = False
            level0_no = 1
            level0_string = str('(main ruleset)')
            yaml_data['(full ruleset)'].append({level0_string: []})
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            letter_no = 0
            letter_sections = False
            notes_no = 0
            bullets_before3 = False
            bullets_before4 = False
            bullets_before5 = False
            letters_before3 = False
            letters_before4 = False
            notes=False
            in_alphabet = False
            in_bullet = False
            penalty4_letter = False
            penalty5_letter = False
            penalty6_letter = False
            ruling_possible = False
            ruling_real = False
            paragraph_no = 1
        if (re.search(r'^(LAW)\s(21)', temp_text)):
            in_laws = False
            break

        # returns defintions or sublaws -- letter sections NOT present --> level 3
        if (re.search(fr'^\(([0-9]|[0-9][0-9])\)\s', temp_text) or re.search(fr'^(F|P)\.K\.\s\(([0-9]|[0-9][0-9])\)\s', temp_text)) and level2_no>=1 and not letter_sections and not in_definitions and in_laws:
            in_alphabet = False
            in_bullet = False
            penalty4 = False
            penalty5 = False
            notes = False
            letters_before3 = False
            bullets_before3 = False
            letters_before4 = False
            bullets_before4 = False
            ruling_possible = False
            ruling_real = False
            level3_no += 1
            level3_string = str('sublaw ' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string: []})
            level4_no = 0
            paragraph_no = 1

        # determines if in notes section
        if re.search(r'^(NOTE|NOTES)\s', temp_text) and not in_definitions and in_laws:
            notes = True
            notes3 = False
            bullets_before3 = False
            bullets_before4 = False
            in_alphabet = False
            paragraph_no = 1

        # returns lowercase letter -- letter sections NOT present --> level 5
        if (re.search(r'^\([a-r]\)\s', temp_text) or re.search(r'^(P|F)\.K\.\s\([a-r]\)\s', temp_text) or re.search(r'^\([0-9]\)\s\([a-r]\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and not letter_sections and not in_definitions and in_laws:
            bullets_before4 = False
            in_bullet = False
            paragraph_no = 1
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                # letters as sublaws of number sublaws
                if not notes and (not penalty4 or not re.search(r'^\(a\)\s', temp_text)) and (not penalty5 or not re.search(r'^\(a\)\s', temp_text)) and not penalty4_letter and not penalty5_letter and (not ruling_real or (ruling_possible and not re.search(r'\(a\)\s', temp_text))):
                    penalty4 = False 
                    penalty5 = False 
                    ruling_possible = False
                    in_alphabet = True
                    if level3_no==0 or letters_before3:
                        letters_before3 = True
                        letters_before3 = True
                        level3_no += 1
                        level3_string = str('subsublaw ' + str(level3_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})
                    else:
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                        penalty5=0
                # for rulings 
                elif ((ruling_possible and re.search(r'\(a\)\s', temp_text)) or ruling_real) and not notes and not penalty4 and not penalty5:
                    ruling_real = True
                    # ruling is same level as numbers, so subruling --> level 4
                    if not in_alphabet and not in_bullet:
                        level4_no += 1
                        level4_string = str('ruling sublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                        level5_no = 0
                    # sublevel of lowercase --> subruling level 5
                    elif in_alphabet and not in_bullet:
                        level5_no += 1
                        level5_string = str('ruling sublaw ' + str(level5_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                        level5_no = 0
                    # sublevel of bullet --> subruling level 6
                    elif in_bullet:
                        level6_no += 1
                        level6_string = str('ruling sublaw ' + str(level6_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                        level7_no = 0
                # for notes
                elif notes:
                    if notes3:
                        level4_no += 1
                        level4_string = str('notes sublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                    elif not notes3:
                        level5_no += 1
                        level5_string = str('notes sublaw ' + str(level5_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                        level6_no = 0
                # for penalty following numbers
                elif penalty4:
                    # makes sure to only enter penalty letter sublist when beginning with a
                    if (not penalty4_letter and re.search(r'\(a\)\s', temp_text)) or penalty4_letter:
                        # no roman numerals in peanlty
                        if not penalty4_roman:
                            if penalty4_letter:
                                level4_no += 1
                                level4_string = str('penalty sublaw ' + str(level4_no))
                                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
                                level4_no = 0
                        # roman numerals in peanlty
                        else: 
                            level5_no += 1
                            level5_string = str('penalty sublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                            level6_no = 0
                        penalty4_letter = True
                # for penalty following lowercase letter sublaw
                elif penalty5:
                    # makes sure to only enter penalty letter sublist when beginning with a
                    if (not penalty5_letter and re.search(r'\(a\)\s', temp_text)) or penalty5_letter:
                        # no roman numerals in penalty
                        if not penalty5_roman:
                            if not penalty5_letter:
                                level5_no += 1
                                level5_string = str('penalty sublaw ' + str(level5_no))
                                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                                level6_no = 0
                        # roman numerals in penalty
                        else:
                            level6_no += 1
                            level6_string = str('penalty sublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
                            level7_no = 0
                        penalty5_letter = True
            elif (re.match(r'\(i\)\s', temp_text) or re.search(r'^(P|F)\.K\.\s\(i\)\s', temp_text) or re.search(r'^\([0-9]\)\s\(i\)\s', temp_text)) and {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string]:
                # letters as sublaws of number sublaws
                if not notes and not penalty4_letter and not penalty5_letter:
                    in_alphabet = True
                    if level3_no==0 or letters_before3:
                        letters_before3 = True
                        level3_no += 1
                        level3_string = str('subsublaw ' + str(level3_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})
                    else:
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                        penalty5=0

        # returns roman numerals (same level as lowercase --> level 5) -- letter sections NOT present 
        notes_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ', 'xi. ', 'xii. ', 'xiii. ', 'xiv. ', 'xv. ', 'xvi. ', 'xvii. ', 'xviii. ', 'xix. ', 'xx. ', 'xxi. ']
        if (re.search(r'\((i|ii|iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text)) and level2_no>=1 and not letter_sections and not in_definitions and in_laws:
            paragraph_no = 1
            # roman numerals in notes
            if notes:
                if level3_no==0 or penalty4 or notes3:
                    notes3 = True
                    level3_no += 1
                    notes_no += 1
                    level3_string = str('notes ' +str(notes_labels[notes_no-1]))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})
                    level4_no = 0
                else:
                    level4_no += 1
                    notes_no += 1
                    level4_string = str('notes ' +str(notes_labels[notes_no-1]))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
            # in penalty following number sublaw
            elif penalty4:
                penalty4_roman = True
                penalty4_letter = False
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
            # in penalty following letter sublaw
            elif penalty5:
                penalty5_roman = True
                penalty5_letter = False
                level5_no += 1
                level5_string = str('penalty ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0

        # returns penalties or exceptions (same level as lowercase --> level 5) -- letter sections NOT present 
        if (re.search(r'^(Penalty:)', temp_text) or re.search(r'^(Exception:|Exceptions:)', temp_text)) and level2_no>=1 and level3_no>=1 and not letter_sections and not in_definitions and in_laws:
            notes = False
            penalty4_roman = False
            penalty4_letter = False
            penalty5_roman = False
            penalty5_letter = False
            ruling_real = False
            ruling_possible = False
            paragraph_no = 1
            # same level as lowercase --> level 4
            if not in_alphabet:
                penalty4 = True
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
            # sublevel of lowercase --> level 5
            elif in_alphabet:
                penalty5 = True
                level5_no += 1
                level5_string = str('penalty ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0

        # returns bullets -- letter sections NOT present 
        if re.search(r'^(o|•|\*)\s', temp_text) and level2_no>=1 and not letter_sections and not in_definitions and in_laws:
            in_bullet = True
            paragraph_no = 1
            # bullets BEFORE numbers --> level 3
            if level3_no==0 or bullets_before3:
                bullets_before3 = True
                level3_no += 1
                level3_string = str('subsubsublaw ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})
            # bullets BEFORE lowercase letters --> level 4
            elif level4_no==0 or bullets_before4:
                bullets_before4 = True
                level4_no += 1
                level4_string = str('subsubsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
            # bullets AFTER lowercase letters or penalties 
            else: 
                # in law itself --> level 5
                if not penalty4 and not penalty5:
                    level5_no += 1
                    level5_string = str('subsubsublaw ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                # for penalties following numerical sublaws
                elif penalty4 and not penalty5:
                    # if no penalty roman numerals or letters
                    if not penalty4_letter and not penalty4_roman:
                        if level4_no>1:
                            level4_no += 1
                            level4_string = str('penalty subsublaw ' + str(level4_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                    # if only penalty letter
                    elif not penalty4_roman and penalty4_roman:
                        if level5_no>1:
                            level5_no += 1
                            level5_string = str('penalty subsublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    # follwoing both roman numerals and lowercase
                    elif penalty4_letter and penalty4_roman:
                        if level6_no>1:
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                # for penalties following lowercase sublaws
                elif penalty5:
                    # if no penalty roman numerals or letters
                    if not penalty5_letter and not penalty5_roman:
                        if level5_no>1:
                            level5_no += 1
                            level5_string = str('penalty subsublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    # if only penalty letter
                    elif not penalty5_roman and penalty5_roman:
                        if level6_no>1:
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    # following both roman numerals and lowercase
                    elif penalty5_letter and penalty5_roman:
                        if level7_no>1:
                            level7_no += 1
                            level7_string = str('penalty subsublaw ' + str(level7_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})

        # returns rulings -- letter sections NOT present 
        if (re.search(r'^(RFU|IRFB)\sRuling', temp_text)) and level2_no>=1 and not letter_sections and not in_definitions and in_laws:
            ruling_possible = True
            penalty4 = False
            penalty5 = False
            paragraph_no = 1
            # same level as numbers --> level 3
            if not in_alphabet and not in_bullet:
                if not notes:
                    level3_no += 1
                    level3_string = str('ruling ' + str(level3_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})
                    level4_no = 0
                else:
                    level4_no += 1
                    level4_string = str('ruling ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
            # sublevel of lowercase --> level 4
            elif in_alphabet and not in_bullet:
                if not notes:
                    level4_no += 1
                    level4_string = str('ruling ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                else:
                    level5_no += 1
                    level5_string = str('ruling ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
            # sublevel of bullet
            elif in_bullet:
                if not notes:
                    level5_no += 1
                    level5_string = str('ruling ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                else:
                    level6_no += 1
                    level6_string = str('ruling ' + str(level6_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0


        # adds paragraphs
        if (re.search(r'^(?!(LAW|Penalt|o|•|\*|-|\(|\s|RFU|IRFB|NOTE|DEF|(P|F)\.K\.)).*', temp_text)) and level2_no>=1 and real_law_no>=1 and len(temp_text) > 40 and not in_definitions and in_laws:
            if level3_no==0 and level4_no==0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph_no += 1
                level0_no += 1
                level0_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'].append({level0_string: []})
            elif level3_no!=0 and level4_no==0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0 and level7_no==0:
                paragraph_no += 1
                level6_no += 1
                level6_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0 and level7_no!=0:
                paragraph_no += 1
                level7_no += 1
                level7_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string: []})


        
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



Run it.

In [ ]:
initial_path_1997_1998 = (TEXTS_DIR / '1997-1998')
final_path_1997_1998 = (SCRUM_DIR / 'clean_')

for file_path in initial_path_1997_1998.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
        year = year.replace('1997-1998', '')
        
        create_yamls_1997_1998(file, final_path_1997_1998, year)

## 1985-1995

Parse the scrum law and save a YAML tree for every file in this range.

In [ ]:
# adding rulings to notes
def create_yamls_1985_1995(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 1
    level2_no = 1
    level3_no = 0
    level4_no = 0
    level5_no = 0
    level6_no = 0
    level7_no = 0
    penalty4 = False
    penalty5 = False
    penalty6 = False
    in_alphabet = False
    in_bullet = False
    notes = False
    in_directives = False
    in_definitions = False
    in_laws = False
    letter_sections = False



    for line in file:
        temp_text = line


        # main ruleset section
        if (not in_definitions or re.search(r'^(LAW)\s1\.', temp_text)) and not in_directives:
            in_definitions = False

        # returns laws --> level 2
        if (re.search(r'^(LAW)\s(20)', temp_text)) and not in_directives:
            level0_no = 1
            level0_string = str('(main ruleset)')
            yaml_data['(full ruleset)'].append({level0_string: []})
            in_laws = True
            in_definitions = False
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            letter_no = 0
            letter_sections = False
            notes_no = 0
            bullets_before3 = False
            bullets_before4 = False
            bullets_before5 = False
            letters_before3 = False
            letters_before4 = False
            notes=False
            notes3 = False
            notes4 = False
            notes5 = False
            in_alphabet = False
            in_bullet = False
            penalty4_letter = False
            penalty5_letter = False
            penalty6_letter = False
            ruling_possible = False
            ruling_real = False
            paragraph_no = 1
        if (re.search(r'^(LAW)\s(21)', temp_text)) and not in_directives:
            in_laws = False
            break

        # returns defintions or sublaws -- letter sections NOT present --> level 3
        if (re.search(fr'^\(([0-9]|[0-9][0-9])\)\s(?!\()', temp_text) or re.search(fr'^(F|P)\.K\.\s\(([0-9]|[0-9][0-9])\)\s', temp_text)) and level2_no>=1 and not letter_sections and not in_directives and not in_definitions and in_laws:
            in_alphabet = False
            in_bullet = False
            penalty4 = False
            penalty5 = False
            notes = False
            letters_before3 = False
            bullets_before3 = False
            letters_before4 = False
            bullets_before4 = False
            ruling_possible = False
            ruling_real = False
            level3_no += 1
            level3_string = str('sublaw ' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string: []})
            level4_no = 0
            level5_no = 0
            level6_no = 0
            paragraph_no = 1

        # determines if in notes section
        if re.search(r'^(NOTE|NOTES)', temp_text) and not in_directives and real_law_no>=1 and not in_definitions and in_laws:
            notes = True
            bullets_before3 = False
            bullets_before4 = False
            in_alphabet = False
            paragraph_no = 1
            # same level as numbers --> level 3
            if not in_alphabet and not in_bullet:
                notes3 = True
                '''level3_no += 1
                level3_string = str('notes ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                level4_no = 0
                level5_no = 0
                level6_no = 0'''
            # sublevel of lowercase --> level 4
            elif (in_alphabet and not in_bullet) or (in_bullet and not in_alphabet):
                notes4 = True
                '''level4_no += 1
                level4_string = str('notes ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0'''
            # sublevel of bullet
            elif in_bullet:
                notes5 = True
                '''level5_no += 1
                level5_string = str('notes ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0'''


        # returns lowercase letter -- letter sections NOT present --> level 5
        if (re.search(r'^\([a-r]\)\s', temp_text) or re.search(r'^(P|F)\.K\.\s\([a-r]\)\s', temp_text) or re.search(r'^\([0-9]\)\s\([a-r]\)\s', temp_text)) and level2_no>=1 and not letter_sections and not in_directives and not in_definitions and in_laws:
            bullets_before4 = False
            in_bullet = False
            paragraph_no = 1
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                # letters as sublaws of number sublaws
                if not notes and (not penalty4 or not re.search(r'^\(a\)\s', temp_text)) and (not penalty5 or not re.search(r'^\(a\)\s', temp_text)) and not penalty4_letter and not penalty5_letter and (not ruling_real or (ruling_possible and not re.search(r'\(a\)\s', temp_text))):
                    penalty4 = False
                    penalty5 = False
                    ruling_possible = False
                    in_alphabet = True
                    if level3_no==0 or letters_before3:
                        letters_before3 = True
                        letters_before3 = True
                        level3_no += 1
                        level3_string = str('subsublaw ' + str(level3_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})
                        level4_no = 0
                        level5_no = 0
                        level6_no = 0
                    else:
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                        level6_no = 0
                        penalty5=0
                # for rulings 
                elif ((ruling_possible and re.search(r'\(a\)\s', temp_text)) or ruling_real) and not notes and not penalty4 and not penalty5:
                    ruling_real = True
                    # ruling is same level as numbers, so subruling --> level 4
                    if not in_alphabet and not in_bullet:
                        level4_no += 1
                        level4_string = str('ruling sublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                        level5_no = 0
                        level6_no = 0
                    # sublevel of lowercase --> subruling level 5
                    elif in_alphabet and not in_bullet:
                        level5_no += 1
                        level5_string = str('ruling sublaw ' + str(level5_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                        level5_no = 0
                        level6_no = 0
                    # sublevel of bullet --> subruling level 6
                    elif in_bullet:
                        level6_no += 1
                        level6_string = str('ruling sublaw ' + str(level6_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                        level7_no = 0
                # for notes
                elif notes:
                    if notes3:
                        level4_no += 1
                        level4_string = str('notes sublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                        level6_no = 0
                    elif not notes3:
                        level5_no += 1
                        level5_string = str('notes sublaw ' + str(level5_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                        level6_no = 0
                # for penalty following numbers
                elif penalty4:
                    # makes sure to only enter penalty letter sublist when beginning with a
                    if (not penalty4_letter and re.search(r'\(a\)\s', temp_text)) or penalty4_letter:
                        # no roman numerals in peanlty
                        if not penalty4_roman:
                            if penalty4_letter:
                                level4_no += 1
                                level4_string = str('penalty sublaw ' + str(level4_no))
                                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
                                level4_no = 0
                                level5_no = 0
                                level6_no = 0
                        # roman numerals in peanlty
                        else: 
                            level5_no += 1
                            level5_string = str('penalty sublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                            level6_no = 0
                        penalty4_letter = True
                # for penalty following lowercase letter sublaw
                elif penalty5:
                    # makes sure to only enter penalty letter sublist when beginning with a
                    if (not penalty5_letter and re.search(r'\(a\)\s', temp_text)) or penalty5_letter:
                        # no roman numerals in penalty
                        if not penalty5_roman:
                            if not penalty5_letter:
                                level5_no += 1
                                level5_string = str('penalty sublaw ' + str(level5_no))
                                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                                level6_no = 0
                        # roman numerals in penalty
                        else:
                            level6_no += 1
                            level6_string = str('penalty sublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
                            level7_no = 0
                        penalty5_letter = True
            elif (re.match(r'\(i\)\s', temp_text) or re.search(r'^(P|F)\.K\.\s\(i\)\s', temp_text) or re.search(r'^\([0-9]\)\s\(i\)\s', temp_text)) and not notes and not penalty4_letter and not penalty5_letter:
                # letters as sublaws of number sublaws
                in_alphabet = True
                if level3_no==0 or letters_before3:
                    if ({'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string]):
                        letters_before3 = True
                        level3_no += 1
                        level3_string = str('subsublaw ' + str(level3_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})
                        level4_no = 0
                        level5_no = 0
                        level6_no = 0
                else:
                    if ({'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string]):
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                        level6_no = 0
                        penalty5=0

        # returns roman numerals (same level as lowercase --> level 5) -- letter sections NOT present 
        notes_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ', 'xi. ', 'xii. ', 'xiii. ', 'xiv. ', 'xv. ', 'xvi. ', 'xvii. ', 'xviii. ', 'xix. ', 'xx. ', 'xxi. ']
        if (re.search(r'\((i|ii|iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text)) and level2_no>=1 and not letter_sections and not in_directives and not in_definitions and in_laws:
            paragraph_no = 1
            # roman numerals in notes
            if notes:
                if level3_no==0 or penalty4 or notes3:
                    notes3 = True
                    level3_no += 1
                    notes_no += 1
                    level3_string = str('notes ' +str(notes_labels[notes_no-1]))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})
                    level4_no = 0
                    level5_no = 0
                    level6_no = 0
                else:
                    level4_no += 1
                    notes_no += 1
                    level4_string = str('notes ' +str(notes_labels[notes_no-1]))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                    level6_no = 0
            # in penalty following number sublaw
            elif penalty4:
                penalty4_roman = True
                penalty4_letter = False
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
            # in penalty following letter sublaw
            elif penalty5:
                penalty5_roman = True
                penalty5_letter = False
                level5_no += 1
                level5_string = str('penalty ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0

        # returns penalties or exceptions (same level as lowercase --> level 5) -- letter sections NOT present 
        if (re.search(r'^(Penalty:)', temp_text) or re.search(r'^(Exception:|Exceptions:)', temp_text)) and level2_no>=1 and level3_no>=1 and not letter_sections and not in_directives and not in_definitions and in_laws:
            notes = False
            penalty4_roman = False
            penalty4_letter = False
            penalty5_roman = False
            penalty5_letter = False
            ruling_real = False
            ruling_possible = False
            paragraph_no = 1
            # same level as lowercase --> level 4
            if not in_alphabet:
                penalty4 = True
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
            # sublevel of lowercase --> level 5
            elif in_alphabet:
                if letters_before3:
                    penalty4 = True
                    level4_no += 1
                    level4_string = str('penalty ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                    level6_no = 0
                else:
                    penalty5 = True
                    level5_no += 1
                    level5_string = str('penalty ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0

        # returns bullets -- letter sections NOT present 
        if re.search(r'^(o|•|\*|-)\s', temp_text) and level2_no>=1 and not letter_sections and not in_directives and not in_definitions and in_laws:
            in_bullet = True
            paragraph_no = 1
            # bullets BEFORE numbers --> level 3
            if level3_no==0 or bullets_before3:
                bullets_before3 = True
                level3_no += 1
                level3_string = str('subsubsublaw ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})
            # bullets BEFORE lowercase letters --> level 4
            elif level4_no==0 or bullets_before4:
                bullets_before4 = True
                level4_no += 1
                level4_string = str('subsubsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
            # bullets AFTER lowercase letters or penalties 
            else: 
                # in law itself --> level 5
                if not penalty4 and not penalty5:
                    penalty4 = False
                    penalty5 = False
                    level5_no += 1
                    level5_string = str('subsubsublaw ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                # for penalties following numerical sublaws
                elif penalty4 and not penalty5:
                    # if no penalty roman numerals or letters
                    if not penalty4_letter and not penalty4_roman:
                        if level4_no>1:
                            level4_no += 1
                            level4_string = str('penalty subsublaw ' + str(level4_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                    # if only penalty letter
                    elif not penalty4_roman and penalty4_letter:
                        if level5_no>1:
                            level5_no += 1
                            level5_string = str('penalty subsublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    # follwoing both roman numerals and lowercase
                    elif penalty4_letter and penalty4_roman:
                        if level6_no>1:
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                # for penalties following lowercase sublaws
                elif penalty5:
                    # if no penalty roman numerals or letters
                    if not penalty5_letter and not penalty5_roman:
                        if level5_no>1:
                            level5_no += 1
                            level5_string = str('penalty subsublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    # if only penalty letter
                    elif not penalty5_roman and penalty5_letter:
                        if level6_no>1:
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    # following both roman numerals and lowercase
                    elif penalty5_letter and penalty5_roman:
                        if level7_no>1:
                            level7_no += 1
                            level7_string = str('penalty subsublaw ' + str(level7_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})

        # returns rulings -- letter sections NOT present 
        if (re.search(r'^(RFU|IRFB)\sRuling', temp_text)) and level2_no>=1 and not letter_sections and not in_directives and not in_definitions and in_laws:
            ruling_possible = True
            penalty4 = False
            penalty5 = False
            paragraph_no = 1
            # same level as numbers --> level 3
            if not in_alphabet and not in_bullet:
                if not notes:
                    level3_no += 1
                    level3_string = str('ruling ' + str(level3_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})
                    level4_no = 0
                    level5_no = 0
                    level6_no = 0
                else:
                    level4_no += 1
                    level4_string = str('ruling ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                    level6_no = 0
            # sublevel of lowercase --> level 4
            elif in_alphabet and not in_bullet:
                if not notes:
                    level4_no += 1
                    level4_string = str('ruling ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                    level6_no = 0
                else:
                    level5_no += 1
                    level5_string = str('ruling ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
            # sublevel of bullet
            elif in_bullet:
                if not notes:
                    level5_no += 1
                    level5_string = str('ruling ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                else:
                    level6_no += 1
                    level6_string = str('ruling ' + str(level6_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0

        # adds paragraphs
        if (re.search(r'^(?!(LAW|Penalt|o|•|\*|-|\(|[0-9]|\s|RFU|IRFB|NOTE|INTER)).*', temp_text)) and level2_no>=1 and real_law_no>=1 and not in_directives and not in_definitions and in_laws:
            if level3_no==0 and level4_no==0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph_no += 1
                level0_no += 1
                level0_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'].append({level0_string: []})
            elif level3_no!=0 and level4_no==0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0 and level7_no==0:
                paragraph_no += 1
                level6_no += 1
                level6_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0 and level7_no!=0:
                paragraph_no += 1
                level7_no += 1
                level7_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string: []})
            

    
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



Run it.

In [ ]:
initial_path_1985_1995 = (TEXTS_DIR / '1985-1995')
final_path_1985_1995 = (SCRUM_DIR / 'clean_')

for file_path in initial_path_1985_1995.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
        
        create_yamls_1985_1995(file, final_path_1985_1995, year)

## 1969-1974

Parse the scrum law and save a YAML tree for every file in this range.

In [ ]:

def create_yamls_1969_1974(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 1
    level0_no = 1
    level2_no = 1
    level3_no = 0
    level4_no = 0
    level5_no = 0
    level6_no = 0
    level7_no = 0
    penalty4 = False
    penalty5 = False
    penalty6 = False
    notes = False
    in_directives = False
    letter_sections = False
    in_definitions = False
    in_laws = False



    for line in file:
        temp_text = line

        # returns laws --> level 2
        if (re.search(r'^(LAW)\s(20)\.', temp_text)) and not in_directives:
            in_laws = True
            in_definitions = False
            level0_no = 1
            level0_string = str('(main ruleset)')
            yaml_data['(full ruleset)'].append({level0_string: []})
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            letter_no = 0
            notes_no = 0
            bullets_before3 = False
            bullets_before4 = False
            bullets_before5 = False
            letters_before3 = False
            letters_before4 = False
            notes=False
            in_alphabet = False
            in_bullet = False
            penalty4_letter = False
            penalty5_letter = False
            penalty6_letter = False
            paragraph_no = 0
        if (re.search(r'^(LAW)\s(21)', temp_text)) and not in_directives:
            in_laws = False
            break

        # returns defintions or sublaws -- letter sections NOT present --> level 3
        if (re.search(fr'^\(([0-9]|[0-9][0-9])\)\s(?!\()', temp_text)) and level2_no>=1 and not in_directives and not in_definitions and in_laws:
            in_alphabet = False
            in_bullet = False
            penalty4 = False
            penalty5 = False
            letters_before3 = False
            bullets_before3 = False
            letters_before4 = False
            bullets_before4 = False
            level3_no += 1
            level3_string = str('sublaw ' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string: []})
            level4_no = 0
            paragraph_no = 1

        # returns lowercase letter -- letter sections NOT present --> level 5
        if (re.search(r'^\([a-r]\)\s', temp_text) or re.search(r'^\([0-9]\)\s\([a-r]\)\s', temp_text)) and level2_no>=1 and not in_directives and not in_definitions and in_laws:
            bullets_before4 = False
            in_bullet = False
            paragraph_no = 1
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                # letters as sublaws of number sublaws
                if (not penalty4 or not re.search(r'^\(a\)\s', temp_text)) and (not penalty5 or not re.search(r'^\(a\)\s', temp_text)) and not penalty4_letter and not penalty5_letter and in_laws:
                    penalty4 = False
                    penalty5 = False
                    in_alphabet = True
                    if level3_no==0 or letters_before3:
                        letters_before3 = True
                        letters_before3 = True
                        level3_no += 1
                        level3_string = str('subsublaw ' + str(level3_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})
                    else:
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                        penalty5=0
                # for penalty following numbers
                elif penalty4:
                    # makes sure to only enter penalty letter sublist when beginning with a
                    if (not penalty4_letter and re.search(r'\(a\)\s', temp_text)) or penalty4_letter:
                        # no roman numerals in peanlty
                        if not penalty4_roman:
                            if penalty4_letter:
                                level4_no += 1
                                level4_string = str('penalty sublaw ' + str(level4_no))
                                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
                                level4_no = 0
                        # roman numerals in peanlty
                        else: 
                            level5_no += 1
                            level5_string = str('penalty sublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                            level6_no = 0
                        penalty4_letter = True
                # for penalty following lowercase letter sublaw
                elif penalty5:
                    # makes sure to only enter penalty letter sublist when beginning with a
                    if (not penalty5_letter and re.search(r'\(a\)\s', temp_text)) or penalty5_letter:
                        # no roman numerals in penalty
                        if not penalty5_roman:
                            if not penalty5_letter:
                                level5_no += 1
                                level5_string = str('penalty sublaw ' + str(level5_no))
                                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                                level6_no = 0
                        # roman numerals in penalty
                        else:
                            level6_no += 1
                            level6_string = str('penalty sublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
                            level7_no = 0
                        penalty5_letter = True
            elif (re.match(r'\(i\)\s', temp_text) or re.search(r'^\([0-9]\)\s\(i\)\s', temp_text)) and {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string]:
                # letters as sublaws of number sublaws
                if not penalty4_letter and not penalty5_letter:
                    in_alphabet = True
                    if level3_no==0 or letters_before3:
                        letters_before3 = True
                        level3_no += 1
                        level3_string = str('subsublaw ' + str(level3_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})
                    else:
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                        penalty5=0

        # returns roman numerals -- letter sections NOT present 
        roman_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ', 'xi. ', 'xii. ', 'xiii. ', 'xiv. ', 'xv. ', 'xvi. ', 'xvii. ', 'xviii. ', 'xix. ', 'xx. ', 'xxi. ']
        # only adds i. if there is a ii., so doesn't add lowercase letter i. 
        if (re.search(r'^\(ii\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and not in_directives and not in_definitions and in_laws:
            paragraph_no = 1
            if penalty4:
                penalty4_roman = True
                penalty4_letter = False
                level4_no += 2
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({'penalty i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({'penalty ii. ':[]})
                level5_no = 0
            elif penalty5:
                penalty5_roman = True
                penalty5_letter = False
                level5_no += 2
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'penalty i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'penalty ii. ':[]})
                level6_no = 0
            else: 
                level5_no += 2
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'ii. ':[]})
                level6_no = 0 
        if (re.search(r'\((iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and level4_no>=1 and not in_directives and not in_definitions and in_laws:
            paragraph_no = 1
            if penalty4:
                penalty4_roman = True
                penalty4_letter = False
                level4_no += 1
                level4_string = roman_labels[level4_no]
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
            elif penalty5:
                penalty5_roman = True
                penalty5_letter = False
                level5_no += 1
                level5_string = roman_labels[level5_no]
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
            else: 
                level5_no += 1
                level5_string = roman_labels[level5_no]
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0 

        # returns penalties or exceptions (same level as lowercase --> level 5) -- letter sections NOT present 
        if (re.search(r'^(Penalty:|Penalties:)', temp_text) or re.search(r'^(Exception:|Exceptions:)', temp_text)) and level2_no>=1 and level3_no>=1 and not in_directives and not in_definitions and in_laws:
            penalty4_roman = False
            penalty4_letter = False
            penalty5_roman = False
            penalty5_letter = False
            paragraph_no = 1
            # same level as lowercase --> level 4
            if not in_alphabet:
                penalty4 = True
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
            # sublevel of lowercase --> level 5
            elif in_alphabet:
                if letters_before3:
                    penalty4 = True
                    level4_no += 1
                    level4_string = str('penalty ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                else:
                    penalty5 = True
                    level5_no += 1
                    level5_string = str('penalty ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0

        # returns bullets -- letter sections NOT present 
        if re.search(r'^(o|•|\*|-)\s', temp_text) and level2_no>=1 and not in_directives and not in_definitions and in_laws:
            in_bullet = True
            paragraph_no = 1
            # bullets BEFORE numbers --> level 3
            if level3_no==0 or bullets_before3:
                bullets_before3 = True
                level3_no += 1
                level3_string = str('subsubsublaw ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})
            # bullets BEFORE lowercase letters --> level 4
            elif level4_no==0 or bullets_before4:
                bullets_before4 = True
                level4_no += 1
                level4_string = str('subsubsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
            # bullets AFTER lowercase letters or penalties 
            else: 
                # in law itself --> level 5
                if not penalty4 and not penalty5:
                    level5_no += 1
                    level5_string = str('subsubsublaw ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                # for penalties following numerical sublaws
                elif penalty4 and not penalty5:
                    # if no penalty roman numerals or letters
                    if not penalty4_letter and not penalty4_roman:
                        if level4_no>1:
                            level4_no += 1
                            level4_string = str('penalty subsublaw ' + str(level4_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                    # if only penalty letter
                    elif not penalty4_roman and penalty4_roman:
                        if level5_no>1:
                            level5_no += 1
                            level5_string = str('penalty subsublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    # follwoing both roman numerals and lowercase
                    elif penalty4_letter and penalty4_roman:
                        if level6_no>1:
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                # for penalties following lowercase sublaws
                elif penalty5:
                    # if no penalty roman numerals or letters
                    if not penalty5_letter and not penalty5_roman:
                        if level5_no>1:
                            level5_no += 1
                            level5_string = str('penalty subsublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    # if only penalty letter
                    elif not penalty5_roman and penalty5_roman:
                        if level6_no>1:
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    # following both roman numerals and lowercase
                    elif penalty5_letter and penalty5_roman:
                        if level7_no>1:
                            level7_no += 1
                            level7_string = str('penalty subsublaw ' + str(level7_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})

        # adds paragraphs
        if (re.search(r'^(?!(LAW|Penalt|o|•|\*|-|\(|[0-9]|\s|NOTE|INTER|A\.|B\.|C\.|D\.|E\.)).*', temp_text)) and not in_directives and in_laws:
            if level3_no==0 and level4_no==0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph_no += 1  
                if paragraph_no >=2:  
                    level0_no += 1
                    level0_string = str(' P' + str(paragraph_no))
                    yaml_data['(full ruleset)'].append({level0_string: []})
            elif level3_no!=0 and level4_no==0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no==0 and level7_no==0:
                paragraph_no += 1
                level5_no += 1
                level5_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0 and level7_no==0:
                paragraph_no += 1
                level6_no += 1
                level6_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0 and level7_no!=0:
                paragraph_no += 1
                level7_no += 1
                level7_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string: []})



        
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



Run it.

In [ ]:
initial_path_1969_1974 = (TEXTS_DIR / '1969-1974')
final_path_1969_1974 = (SCRUM_DIR / 'clean_')

for file_path in initial_path_1969_1974.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')

        create_yamls_1969_1974(file, final_path_1969_1974, year)

## 1954-1959

Parse the scrum law and save a YAML tree for every file in this range.

In [ ]:
def create_yamls_1954_1959(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 1
    level1_no = 1
    level2_no = 1
    level3_no = 0
    level4_no = 0
    level5_no = 0
    level6_no = 0
    penalty4 = False
    penalty5 = False
    penalty6 = False
    in_directives = False
    letter_sections = False
    in_laws = False



    for line in file:
        temp_text = line

        # returns laws --> level 2
        if (re.search(r'^(LAW)\s(15)', temp_text)) and not in_directives:
            in_laws = True
            level0_no = 1
            level0_string = str('(main ruleset)')
            yaml_data['(full ruleset)'].append({level0_string: []})
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            letter_no = 0
            letter_sections = False
            bullets_before3 = False
            bullets_before4 = False
            bullets_before5 = False
            letters_before3 = False
            letters_before4 = False
            in_alphabet = False
            in_bullet = False
            penalty4 = False
            penalty5 = False
            penalty6 = False
            paragraph_no = 0
        if (re.search(r'^(LAW)\s(16)', temp_text)) and not in_directives:
            in_laws = False
            break

        # returns letter sections --> level 3 if present
        if re.search(fr'^\((A|B|C|D|E)\)\s', temp_text) and level2_no>=1 and not in_directives and in_laws:
            letter_sections = True
            letter_no += 1
            level3_no += 1
            level3_string = str('LETTER ' + str(letter_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string: []})
            level4_no = 0
            penalty4 = False
            penalty5 = False
            paragraph_no = 1

        # returns defintions or sublaws -- letter sections present --> level 4
        if (re.search(fr'^\(([0-9]|[0-9][0-9])\)\s(?!\()', temp_text)) and level2_no>=1 and level3_no>=1 and letter_sections and not in_directives and in_laws:
            in_alphabet = False
            in_bullet = False
            penalty5 = False
            penalty6 = False
            letters_before4 = False
            bullets_before4 = False
            letters_before5 = False
            bullets_before5 = False
            ruling_possible = False
            ruling_real = False
            level4_no += 1
            level4_string = str('sublaw ' + str(level4_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
            level5_no = 0
            paragraph_no = 1

        # returns lowercase letter -- letter sections present --> level 5
        if (re.search(r'^\([a-r]\)\s', temp_text) or re.search(r'^(P|F)\.K\.\s\([a-r]\)\s', temp_text) or re.search(r'^\([0-9]\)\s\([a-r]\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and letter_sections and not in_directives and in_laws:
            bullets_before5 = False
            in_bullet = False
            paragraph_no = 1
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                # letters as sublaws of number sublaws
                in_alphabet = True
                if level4_no==0 or letters_before4:
                    letters_before4 = True
                    letters_before4 = True
                    level4_no += 1
                    level4_string = str('subsublaw ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                else:
                    level5_no += 1
                    level5_string = str('subsublaw ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                    level6_no = 0
                    level6_no=0
            elif (re.match(r'\(i\)\s', temp_text)):
                # letters as sublaws of number sublaws
                in_alphabet = True
                paragraph_no = 1
                if level4_no==0 or letters_before4:
                    if {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string]:
                        letters_before4 = True
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                else:
                    if {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string]: 
                        level5_no += 1
                        level5_string = str('subsublaw ' + str(level5_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                        level6_no = 0
                        level6_no=0    

        # returns roman numerals (same level as lowercase --> level 5) -- letter sections present 
        roman_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ', 'xi. ', 'xii. ', 'xiii. ', 'xiv. ', 'xv. ', 'xvi. ', 'xvii. ', 'xviii. ', 'xix. ', 'xx. ', 'xxi. ']
        # only adds i. if there is a ii., so doesn't add lowercase letter i. 
        if (re.search(r'^\(ii\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and level4_no>=1 and letter_sections and not in_directives and in_laws:
            paragraph_no = 1
            if penalty5:
                penalty5_roman = True
                level5_no += 1
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'penalty ii. ':[]})
                level6_no = 0
            elif penalty6:
                penalty6_roman = True
                level6_no += 1
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({'penalty ii. ':[]})
                level7_no = 0
            else: 
                if letters_before4:
                    level5_no += 2
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'i. ':[]})
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'ii. ':[]})
                    level6_no = 0 
                else:
                    level6_no += 2
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({'i. ':[]})
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({'ii. ':[]})
                    level7_no = 0 
        if (re.search(r'\((iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and level4_no>=1 and letter_sections and not in_directives and in_laws:
            paragraph_no = 1
            if penalty5:
                penalty5_roman = True
                level5_no += 1
                level5_string = str('penalty' + str(roman_labels[level5_no]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
            elif penalty6:
                penalty6_roman = True
                level6_no += 1
                level6_string = str('penalty ' + str(roman_labels[level6_no]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
            else: 
                if letters_before4:
                    level5_no += 1
                    level5_string = roman_labels[level5_no]
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                else:
                    level6_no += 1
                    level6_string = roman_labels[level6_no]
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0 

        # returns penalties or exceptions (same level as lowercase --> level 6) -- letter sections present 
        if (re.search(r'^(Penalty:)', temp_text) or re.search(r'^(Exception:|Exceptions:)', temp_text)) and level2_no>=1 and level3_no>=1 and letter_sections and not in_directives and in_laws:
            penalty5_roman = False
            penalty6_roman = False
            paragraph_no = 1
            # no sub-points --> level 4
            if level4_no==0:
                penalty4 = True
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
            # same level as lowercase --> level 5
            elif not in_alphabet:
                penalty5 = True
                level5_no += 1
                level5_string = str('penalty ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
            # sublevel of lowercase --> level 6
            elif in_alphabet:
                if letters_before4:
                    penalty5 = True
                    level5_no += 1
                    level5_string = str('penalty ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                else: 
                    penalty6 = True
                    level6_no += 1
                    level6_string = str('penalty ' + str(level6_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0
        
        # number sublist --> follows alphabet -- letter sections present
        if (re.search(r'^([1-9])\.\s', temp_text)) and level2_no>=1 and level3_no>=1 and level4_no>=1 and letter_sections and not in_directives and in_laws:
            paragraph_no = 1
            if penalty5:
                penalty5_roman = True
                level5_no += 1
                level5_string = str('number item ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
            elif penalty6:
                penalty6_roman = True
                level6_no += 1
                level6_string = str('number item ' + str(level6_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
            else: 
                if letters_before4:
                    level5_no += 1
                    level5_string = str('number item ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                else:
                    level6_no += 1
                    level6_string = str('number item ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0

        # returns bullets -- letter sections present 
        if re.search(r'^(o|•|\*|-)\s', temp_text) and level2_no>=1 and level3_no>=1 and letter_sections and not in_directives and in_laws:
            in_bullet = True
            paragraph_no = 1
            # bullets BEFORE numbers --> level 4
            if level4_no==0 or bullets_before4:
                bullets_before4 = True
                level4_no += 1
                level4_string = str('subsubsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
            # bullets BEFORE lowercase letters --> level 5
            elif level5_no==0 or bullets_before5:
                bullets_before5 = True
                level5_no += 1
                level5_string = str('subsubsublaw ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
            # bullets AFTER lowercase letters or penalties 
            else: 
                # in law itself --> level 6
                if not penalty5 and not penalty6:
                    level6_no += 1
                    level6_string = str('subsubsublaw ' + str(level6_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                # for penalties following numerical sublaws
                elif penalty5 and not penalty6:
                    # if no penalty roman numerals or letters
                    if not penalty5_roman:
                        if level5_no>1:
                            level5_no += 1
                            level5_string = str('penalty subsublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    # follwoing both roman numerals and lowercase
                    elif penalty5_roman:
                        if level6_no>1:
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                # for penalties following lowercase sublaws
                elif penalty6:
                    # if no penalty roman numerals or letters
                    if not penalty6_roman:
                        if level6_no>1:
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    # following both roman numerals and lowercase
                    elif penalty6_roman:
                        if level7_no>1:
                            level7_no += 1
                            level7_string = str('penalty subsublaw ' + str(level7_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})





        # returns defintions or sublaws -- letter sections NOT present --> level 3
        if (re.search(fr'^\(([0-9]|[0-9][0-9])\)\s(?!\()', temp_text) or re.search(fr'^(F|P)\.K\.\s\(([0-9]|[0-9][0-9])\)\s(?!\()', temp_text)) and level2_no>=1 and not letter_sections and not in_directives and in_laws:
            in_alphabet = False
            in_bullet = False
            penalty4 = False
            penalty5 = False
            letters_before3 = False
            bullets_before3 = False
            letters_before4 = False
            bullets_before4 = False
            level3_no += 1
            level3_string = str('sublaw ' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string: []})
            level4_no = 0
            paragraph_no = 1

        # returns lowercase letter -- letter sections NOT present --> level 5
        if (re.search(r'^\([a-r]\)\s', temp_text)) and level2_no>=1 and not letter_sections and not in_directives and in_laws:
            bullets_before4 = False
            in_bullet = False
            paragraph_no = 1
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                # letters as sublaws of number sublaws
                if (not penalty4 or not re.search(r'^\(a\)\s', temp_text)) and (not penalty5 or not re.search(r'^\(a\)\s', temp_text)):
                    penalty4 = False
                    penalty5 = False 
                    in_alphabet = True
                    if level3_no==0 or letters_before3:
                        letters_before3 = True
                        letters_before3 = True
                        level3_no += 1
                        level3_string = str('subsublaw ' + str(level3_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})
                    else:
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                        penalty5=0
            elif (re.match(r'\(i\)\s', temp_text)):
                # letters as sublaws of number sublaws
                in_alphabet = True
                if level3_no==0 or letters_before3:
                    if {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string]:
                        letters_before3 = True
                        level3_no += 1
                        level3_string = str('subsublaw ' + str(level3_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})
                else:
                    if {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string]:
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                        penalty5=0

        
        # returns roman numerals (same level as lowercase --> level 4) -- letter sections NOT present 
        roman_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ', 'xi. ', 'xii. ', 'xiii. ', 'xiv. ', 'xv. ', 'xvi. ', 'xvii. ', 'xviii. ', 'xix. ', 'xx. ', 'xxi. ']
        # only adds i. if there is a ii., so doesn't add lowercase letter i. 
        if (re.search(r'^\(ii\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and not letter_sections and not in_directives and in_laws:
            paragraph_no = 1
            if penalty4:
                penalty4_roman = True
                level4_no += 1
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({'penalty ii. ':[]})
                level5_no = 0
            elif penalty5:
                penalty5_roman = True
                level5_no += 1
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'penalty ii. ':[]})
                level6_no = 0
            else: 
                if letters_before3:
                    level4_no += 2
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({'i. ':[]})
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({'ii. ':[]})
                    level5_no = 0
                else:
                    level5_no += 2
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'i. ':[]})
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'ii. ':[]})
                    level6_no = 0 
        if (re.search(r'\((iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and not letter_sections and not in_directives and in_laws:
            paragraph_no = 1
            if penalty4:
                penalty4_roman = True
                level4_no += 1
                level4_string = str('penalty ' + str(roman_labels[level4_no]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
            elif penalty5:
                penalty5_roman = True
                level5_no += 1
                level5_string = str('penalty' + str(roman_labels[level5_no]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
            else: 
                if letters_before3:
                    level4_no += 1
                    level4_string = roman_labels[level5_no]
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0 
                else:
                    level5_no += 1
                    level5_string = roman_labels[level5_no]
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0 

        # returns penalties or exceptions (same level as lowercase --> level 5) -- letter sections NOT present 
        if (re.search(r'^(Penalty:)', temp_text) or re.search(r'^(Exception:|Exceptions:)', temp_text)) and level2_no>=1 and not letter_sections and not in_directives and in_laws:
            penalty4_roman = False
            penalty5_roman = False
            paragraph_no = 1
            # no sub-points --> level 3
            if level3_no==0:
                penalty3 = True
                level3_no += 1
                level3_string = str('penalty ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})
                level4_no = 0
            # same level as lowercase --> level 4
            elif not in_alphabet:
                penalty4 = True
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
            # sublevel of lowercase --> level 5
            elif in_alphabet:
                if letters_before3:
                    penalty4 = True
                    level4_no += 1
                    level4_string = str('penalty ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                else:
                    penalty5 = True
                    level5_no += 1
                    level5_string = str('penalty ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
        
        # number sublist --> follows alphabet -- letter sections present
        if (re.search(r'^([1-9])\.\s', temp_text)) and level2_no>=1 and level3_no>=1 and not letter_sections and not in_directives and in_laws:
            paragraph_no = 1
            if penalty4:
                penalty4_roman = True
                level4_no += 1
                level4_string = str('number item ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
            elif penalty5:
                penalty5_roman = True
                level5_no += 1
                level5_string = str('number item ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
            else: 
                if letters_before3:
                    level4_no += 1
                    level4_string = str('number item ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                else:
                    level5_no += 1
                    level5_string = str('number item ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0   


        # returns bullets -- letter sections NOT present 
        if re.search(r'^(o|•|\*|-)\s', temp_text) and level2_no>=1 and not letter_sections and not in_directives and in_laws:
            in_bullet = True
            paragraph_no = 1
            # bullets BEFORE numbers --> level 3
            if level3_no==0 or bullets_before3:
                bullets_before3 = True
                level3_no += 1
                level3_string = str('subsubsublaw ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})
            # bullets BEFORE lowercase letters --> level 4
            elif level4_no==0 or bullets_before4:
                bullets_before4 = True
                level4_no += 1
                level4_string = str('subsubsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
            # bullets AFTER lowercase letters or penalties 
            else: 
                # in law itself --> level 5
                if not penalty4 and not penalty5:
                    level5_no += 1
                    level5_string = str('subsubsublaw ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                # for penalties following numerical sublaws
                elif penalty4 and not penalty5:
                    # if no penalty roman numerals or letters
                    if not penalty4_roman:
                        if level4_no>1:
                            level4_no += 1
                            level4_string = str('penalty subsublaw ' + str(level4_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                    # if only penalty letter
                    elif penalty4_roman:
                        if level5_no>1:
                            level5_no += 1
                            level5_string = str('penalty subsublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
               # for penalties following lowercase sublaws
                elif penalty5:
                    # if no penalty roman numerals or letters
                    if not penalty5_roman:
                        if level5_no>1:
                            level5_no += 1
                            level5_string = str('penalty subsublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    # if only penalty letter
                    elif penalty5_roman:
                        if level6_no>1:
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})

        # adds paragraphs
        if (re.search(r'^(?!(LAW|Penalty|o|•|\*|-|\(|[0-9]|\s|NOTE|A\s|B\s|C\s|D\s|E\s)).*', temp_text)) and level2_no>=1 and real_law_no>=1 and not in_directives and in_laws:
            if level3_no==0 and level4_no==0 and level5_no==0 and level6_no==0:
                paragraph_no += 1  
                if paragraph_no >=2:  
                    level0_no += 1
                    level0_string = str(' P' + str(paragraph_no))
                    yaml_data['(full ruleset)'].append({level0_string: []})
            elif level3_no!=0 and level4_no==0 and level5_no==0 and level6_no==0:
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no==0 and level6_no==0:
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no==0:
                paragraph_no += 1
                level5_no += 1
                level5_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0:
                paragraph_no += 1
                level6_no += 1
                level6_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})

            
                
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



Run it.

In [ ]:
initial_path_1954_1959 = (TEXTS_DIR / '1954-1959')
final_path_1954_1959 = (SCRUM_DIR / 'clean_')

for file_path in initial_path_1954_1959.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
        
        create_yamls_1954_1959(file, final_path_1954_1959, year)

## 1934-1949

Parse the scrum law and save a YAML tree for every file in this range.

In [ ]:

def create_yamls_1934_1949(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 1
    level1_no = 1
    level2_no = 1
    level3_no = 0
    level4_no = 0
    level5_no = 0
    level6_no = 0
    penalty4 = False
    penalty5 = False
    penalty6 = False
    in_directives = False
    in_laws = False



    for line in file:
        temp_text = line


        # returns laws --> level 2
        if (re.search(r'^(LAW)\s(15)', temp_text)) and not in_directives:
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            numbers_before3 = False
            in_numbers = False
            penalty3 = False
            penalty4 = False
            penalty5 = False
            notes3 = False
            paragraph_no = 0
            level0_no = 1
            level0_string = str('(main ruleset)')
            yaml_data['(full ruleset)'].append({level0_string: []})
            in_laws = True
        if (re.search(r'^(LAW)\s(16)', temp_text)) and not in_directives:
            in_laws = False
            break


        # returns lowercase letter -- letter sections NOT present --> level 3 + 4
        if (re.search(r'^\([a-r]\)\s', temp_text)) and level2_no>=1 and not in_directives and in_laws:
            in_numbers = False
            notes3 = False
            paragraph_no = 1
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                penalty4 = False
                penalty5 = False
                numbers_before3 = False
                level3_no += 1
                level3_string = str('sublaw ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string: []})
                level4_no = 0
            elif (re.match(r'\(i\)\s', temp_text)) and not penalty3 and not penalty4 and not penalty5:
                penalty4 = False
                penalty5 = False
                numbers_before3 = False
                level3_no += 1
                level3_string = str('sublaw ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string: []})
                level4_no = 0


        # returns defintions or sublaws -- letter sections NOT present --> level 3
        if (re.search(fr'^\(([0-9]|[0-9][0-9])\)\s(?!\()', temp_text)) and level2_no>=1 and not in_directives and in_laws:
            notes3 = False
            paragraph_no = 1
            if (not penalty3 or (penalty3 and not re.search(r'^\(1\)\s', temp_text))) and (not penalty4 or (penalty4 and not re.search(r'^\(1\)\s', temp_text)))and (not penalty5 or (penalty5 and not re.search(r'^\(1\)\s', temp_text))):
                penalty3 = False
                penalty4 = False
                penalty5 = False
                in_numbers = True
                if level3_no==0 or numbers_before3:
                    numbers_before3 = True
                    numbers_before3 = True
                    level3_no += 1
                    level3_string = str('subsublaw ' + str(level3_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})
                else:
                    level4_no += 1
                    level4_string = str('subsublaw ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
                    level5_no = 0
                    penalty5=0
            elif penalty3:
                level3_no += 1
                level3_string = str('penalty ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})
                level4_no = 0
            elif penalty4:
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
            elif penalty5:
                level5_no += 1
                level5_string = str('penalty' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0

        
        # returns roman numerals (same level as lowercase --> level 4) -- letter sections NOT present 
        roman_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ', 'xi. ', 'xii. ', 'xiii. ', 'xiv. ', 'xv. ', 'xvi. ', 'xvii. ', 'xviii. ', 'xix. ', 'xx. ', 'xxi. ']
        # only adds i. if there is a ii., so doesn't add lowercase letter i. 
        if (re.search(r'^\(ii\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and not in_directives and in_laws:
            paragraph_no = 1
            if penalty3:
                level3_no += 1
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({'penalty ii. ':[]})
                level4_no = 0
            elif penalty4:
                level4_no += 1
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({'penalty ii. ':[]})
                level5_no = 0
            elif penalty5:
                level5_no += 1
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'penalty ii. ':[]})
                level6_no = 0
        if (re.search(r'\((iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and not in_directives and in_laws:
            paragraph_no = 1
            if penalty3:
                level3_no += 1
                level3_string = str('penalty ' + str(roman_labels[level3_no]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})
                level4_no = 0
            elif penalty4:
                level4_no += 1
                level4_string = str('penalty ' + str(roman_labels[level4_no]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
            elif penalty5:
                level5_no += 1
                level5_string = str('penalty' + str(roman_labels[level5_no]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0


        # returns penalties  (same level as lowercase --> level 5) -- letter sections NOT present 
        if (re.search(r'^(Penalty:)', temp_text)) and level2_no>=1 and not in_directives and in_laws:
            paragraph_no = 1
            # no sub-points --> level 3
            if level3_no==0:
                penalty3 = True
                level3_no += 1
                level3_string = str('penalty ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})
                level4_no = 0
            # same level as lowercase --> level 4
            elif not in_numbers:
                penalty4 = True
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
            # sublevel of lowercase --> level 5
            elif in_numbers:
                if numbers_before3:
                    penalty4 = True
                    level4_no += 1
                    level4_string = str('penalty ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                else:
                    penalty5 = True
                    level5_no += 1
                    level5_string = str('penalty ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0  

        # returns notes 
        if (re.search(r'^(Note:)', temp_text)) and level2_no>=1 and not in_directives and in_laws:
            paragraph_no = 1
            # no sub-points --> level 3
            if level3_no==0 or notes3:
                notes3 = True
                level3_no += 1
                level3_string = str('note ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})
                level4_no = 0
            # same level as lowercase --> level 4
            elif not in_numbers:
                level4_no += 1
                level4_string = str('note ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
            # sublevel of lowercase --> level 5
            elif in_numbers:
                if numbers_before3:
                    level4_no += 1
                    level4_string = str('note ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                else:
                    level5_no += 1
                    level5_string = str('note ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0 

        # adds paragraphs
        if (re.search(r'^(?!(Part|LAW|Penalty|i|v|\(|[0-9]|\s|NOTES|Note)).*', temp_text)) and level2_no>=1 and real_law_no>=1 and not in_directives and in_laws:
            if level3_no==0 and level4_no==0 and level5_no==0:
                paragraph_no += 1
                level0_no += 1
                level0_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'].append({level0_string: []})
            elif level3_no!=0 and level4_no==0 and level5_no==0:
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no==0:
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0:
                paragraph_no += 1
                level5_no += 1
                level5_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
            '''elif level3_no!=0 and level4_no!=0 and level5_no!=0:
                paragraph_no += 1
                level6_no += 1
                level6_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})'''


       
    yaml_data['(full ruleset)'][0]['(main ruleset)'].append({'Law 1 ': []})
    
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



Run it.

In [ ]:
initial_path_1934_1949 = (TEXTS_DIR / '1934-1949')
final_path_1934_1949 = (SCRUM_DIR / 'clean_')

for file_path in initial_path_1934_1949.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
        #print(year)
        create_yamls_1934_1949(file, final_path_1934_1949, year)

## 1929

Parse the scrum law and save a YAML tree for every file in this range.

In [ ]:
def create_yamls_1929(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 1
    level1_no = 1
    level2_no = 1
    level3_no = 0
    level4_no = 0
    penalty4 = False
    penalty5 = False
    penalty6 = False
    in_directives = False
    in_laws = False

    for line in file:
        temp_text = line

        # returns laws --> level 2
        if (re.search(r'^(LAW)\s(15)\.', temp_text)) and not in_directives:
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            penalty3 = False
            penalty4 = False
            paragraph_no = 1
            level0_no = 1
            level0_string = str('(main ruleset)')
            yaml_data['(full ruleset)'].append({level0_string: []})
            in_laws = True
        if (re.search(r'^(LAW)\s(16)', temp_text)) and not in_directives:
            in_laws = False
            break

        # returns lowercase letter -- letter sections NOT present --> level 3 + 4
        if (re.search(r'^\([a-r]\)\s', temp_text)) and level2_no>=1 and not in_directives and in_laws:
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                if (not penalty3 or (penalty3 and not re.search(r'^\(a\)\s', temp_text))) and (not penalty4 or (penalty4 and not re.search(r'^\(a\)\s', temp_text))):
                    penalty3 = False
                    penalty4 = False
                    numbers_before3 = False
                    level3_no += 1
                    level3_string = str('sublaw ' + str(level3_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string: []})
                    level4_no = 0

                    paragraph_no = 1

                elif penalty3 and (penalty_letters or re.search(r'^\(a\)\s', temp_text)):
                    penalty_letters = True
                    if not re.search(r'^\(a\)\s', temp_text):
                        level3_no += 1
                        level3_string = str('penalty ' + str(level3_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string: []})
                        level4_no = 0

                elif penalty4 and (penalty_letters or re.search(r'^\(a\)\s', temp_text)):
                    penalty_letters = True
                    if not re.search(r'^\(a\)\s', temp_text):
                        level4_no += 1
                        level4_string = str('penalty ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0

            elif (re.match(r'\(i\)\s', temp_text)) and not penalty3 and not penalty4 and not penalty5:
                if not penalty3 and not penalty4:
                    numbers_before3 = False
                    level3_no += 1
                    level3_string = str('sublaw ' + str(level3_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string: []})
                    level4_no = 0

                    paragraph_no = 1

                elif penalty3 and (penalty_letters):
                    level3_no += 1
                    level3_string = str('penalty ' + str(level3_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string: []})
                    level4_no = 0

                elif penalty4 and penalty_letters:
                    level4_no += 1
                    level4_string = str('penalty ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
                    level5_no = 0


        
        # returns roman numerals (same level as lowercase --> level 4) -- letter sections NOT present 
        roman_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ', 'xi. ', 'xii. ', 'xiii. ', 'xiv. ', 'xv. ', 'xvi. ', 'xvii. ', 'xviii. ', 'xix. ', 'xx. ', 'xxi. ']
        # only adds i. if there is a ii., so doesn't add lowercase letter i. 
        if (re.search(r'^\(ii\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and not in_directives and in_laws:
            if penalty3:
                level3_no += 1
                level3_string = str('penalty ii. ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})
                level4_no = 0

                paragraph_no = 1

            elif penalty4:
                level4_no += 1
                level4_string = str('penalty ii. ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({'penalty ii. ':[]})
                level5_no = 0

                paragraph_no = 1

        if (re.search(r'\((iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and not in_directives and in_laws:
            if penalty3:
                level3_no += 1
                level3_string = str('penalty ' + str(roman_labels[level3_no]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})
                level4_no = 0

                paragraph_no = 1

            elif penalty4:
                level4_no += 1
                level4_string = str('penalty ' + str(roman_labels[level4_no]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0

                paragraph_no = 1



        # returns penalties -- letter sections NOT present 
        if (re.search(r'^(Penalty:)', temp_text)) and level2_no>=1 and not in_directives and in_laws:
            penalty_letters = False
            # no sub-points --> level 3
            if level3_no==0 or penalty3:
                penalty3 = True
                level3_no += 1
                level3_string = str('penalty ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})
                level4_no = 0

            # sublevel of lowercase --> level 4
            else:
                penalty4 = True
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0


        # adds paragraphs
        if (re.search(r'^(?!(LAW|Penalty|i|v|\(|[0-9]|\s|CIRCULAR)).*', temp_text)) and level2_no>=1 and real_law_no>=1 and not in_directives and in_laws:
            if level3_no==0 and level4_no==0:
                paragraph_no += 1
                level0_no += 1
                level0_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'].append({level0_string: []})
            elif level3_no!=0 and level4_no==0:
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string: []})
            elif level3_no!=0 and level4_no!=0:
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
            '''elif level3_no!=0 and level4_no!=0:
                paragraph_no += 1
                level5_no += 1
                level5_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})'''

    
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



Run it.

In [ ]:
initial_path_1929 = (TEXTS_DIR / '1929')
final_path_1929 = (SCRUM_DIR / 'clean_')

for file_path in initial_path_1929.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
        
        create_yamls_1929(file, final_path_1929, year)

## 1894-1924

Parse the scrum law and save a YAML tree for every file in this range.

In [ ]:
def create_yamls_1894_1924(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 1
    level1_no = 1
    level2_no = 1
    level3_no = 0
    level4_no = 0
    notes3 = False
    in_directives = False
    in_laws = False


    for line in file:
        temp_text = line

         # returns laws --> level 2
        if (re.search(r'^(A)\s(scrummage)', temp_text, flags=re.IGNORECASE) and not in_directives):
            level0_no = 1
            level0_string = str('(main ruleset)')
            yaml_data['(full ruleset)'].append({level0_string: []})
            level3_no = 0
            level4_no = 0
            notes3 = False
            paragraph_no = 1
            in_laws = True
        if (re.search(r'^(A)\s(try)', temp_text, flags=re.IGNORECASE)):
            in_laws=False
            break

        # returns lowercase letter -- letter sections NOT present --> level 3 + 4
        if (re.search(r'^(\([a-r]\)|\([a-r]\.\))\s', temp_text)) and level2_no>=1 and not in_directives and in_laws:
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
                notes3 = False
                level3_no += 1
                level3_string = str('sublaw ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string: []})
                level4_no = 0
                paragraph_no = 1


        # returns notes -- letter sections NOT present 
        if (re.search(r'^((Note)|(NOTE))\.', temp_text)) and level2_no>=1 and not in_directives and in_laws:
            paragraph_no = 1
            # no sub-points --> level 3
            if level3_no==0 or notes3:
                notes3 = True
                level3_no += 1
                level3_string = str('notes ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})
                level4_no = 0
            # sublevel of lowercase --> level 4
            else:
                level4_no += 1
                level4_string = str('notes ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
        
        if (re.search(r'^(?!(Note|NOTE|\#|\*|I|V|\(|[0-9]|\s|INTER|At)).*', temp_text)) and real_law_no>=1 and not in_directives and in_laws:
            if level3_no==0 and level4_no==0:
                paragraph_no += 1
                level0_no += 1
                level0_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'].append({level0_string: []})
            elif level3_no!=0 and level4_no==0:
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string: []})
            elif level3_no!=0 and level4_no!=0:
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
            '''elif level3_no!=0 and level4_no!=0:
                paragraph_no += 1
                level5_no += 1
                level5_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})'''


 
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



Run it.

In [ ]:
initial_path_1894_1924 = (TEXTS_DIR / '1894-1924')
final_path_1894_1924 = (SCRUM_DIR / 'clean_')

for file_path in initial_path_1894_1924.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        create_yamls_1894_1924(file, final_path_1894_1924, year)

## 1874-1889

Parse the scrum law and save a YAML tree for every file in this range.

In [ ]:
def create_yamls_1874_1889(file, save_path, year):
    yaml_data = {'(full ruleset)': []}

    level0_no = 1
    level0_string = str('(main ruleset)')
    yaml_data['(full ruleset)'].append({level0_string: []})

    level3_no = 1
    level3_string = str('text')
    yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string:[]})

    
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



Run it.

In [ ]:
initial_path_1874_1889 = (TEXTS_DIR / '1874-1889')
final_path_1874_1889 = (SCRUM_DIR / 'clean_')

for file_path in initial_path_1874_1889.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        create_yamls_1874_1889(file, final_path_1874_1889, year)

## Comparing trees across years

`generate_graph_comparison` draws every saved tree from a folder into one
grid figure (11 rows x 4 columns).

In [ ]:
def generate_graph_comparison(file_path):

    fig, axes = plt.subplots(nrows = 11, ncols = 4, figsize=(28,77))
    flat_axes = axes.flatten()
    
    for i, file_path in enumerate(sorted(file_path.glob('*.yaml'))):
        with open(file_path, 'r') as file:
            file_name = file_path.stem
            year = file_name.replace('tree', '')
            year = year.replace('clean_', '')

            data = yaml.safe_load(file)
            G = create_heirarchical_tree(data)

            pos = graphviz_layout(G, prog='dot', args='-Gnodesep=0.4 -Granksep=75')
            node_colors = [G.nodes[node]['color'] for node in G.nodes()]
            node_sizes = [G.nodes[node]['node_size'] for node in G.nodes()]

            ax = flat_axes[i]

            nx.draw(G, pos, ax=ax, node_size=node_sizes, node_color=node_colors, width=1, edge_color='dimgrey', arrows=False)

            # make size of axes so doesn't pull diagram
            xs = [p[0] for p in pos.values()]
            ys = [p[1] for p in pos.values()]
            max_range = max(max(xs) - min(xs), max(ys) - min(ys))
            center_x = (min(xs) + max(xs)) / 2
            center_y = (min(ys) + max(ys)) / 2
            padding = 0.05 * max_range
            half = (max_range / 2) + padding
            ax.set_xlim(center_x - half, center_x + half)
            ax.set_ylim(center_y - half, center_y + half)
            ax.set_aspect("equal")

            ax.set_title(str(year), size=30)

    plt.tight_layout()
    return fig, axes


Run it on the saved scrum trees.

In [ ]:
generate_graph_comparison(SCRUM_DIR)

Reload every saved tree and count "terminating" (leaf, out-degree 0)
nodes per year.

In [ ]:
scrum_path = SCRUM_DIR

scrum_trees = {}
term_nodes = {}

for file_path in (scrum_path.glob('*.yaml')):
    with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
        file_name = file_path.stem
        year = file_name.replace('tree', '')
        year = year.replace('clean_', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)
        
        scrum_trees[int(year)] = G
        term_nodes[int(year)] = sum(1 for node in G if G.out_degree(node)==0)


Plot terminating-node count vs. year (log y-axis).

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(term_nodes.keys(), term_nodes.values(), color='black')
plt.ylabel('# terminating nodes', size=16)
plt.xlabel('Year', size=16)
plt.yscale('log')


## Terminating nodes vs. cumulative matches

Same cumulative-matches CSV used in the other notebooks.

In [ ]:
csv_path = MATCHES_CSV_PATH

years_matches = {}

with open(csv_path, 'r') as csvfile: # adds element to dictionary of year: total matches
    reader = csv.DictReader(csvfile)
    for row in reader:
        years_matches[int(row['year'])] = int(row['n_matches_sum'])

Convert terminating-node counts to be keyed by cumulative matches (from 1871 onward).

In [ ]:
term_nodes_m = {}

# make dictionary cumulative matches: words
for year in term_nodes: 
    # cutting out first 25 years where test matches not recorded
    if year >= 1871: 
        matches_temp = years_matches[year]

    term_nodes_m[matches_temp] = term_nodes[year]

Plot terminating-node count vs. cumulative matches (log-log).

In [ ]:
plt.figure(figsize=(5,5))
#plt.scatter(np.power(list(term_nodes_m.keys()), 1/2), term_nodes_m.values(), color='black')
plt.scatter(term_nodes_m.keys(), term_nodes_m.values(), color='black')
plt.ylabel('# terminating nodes', size=16)
plt.xlabel('# matches', size=16)
plt.xscale('log')
plt.yscale('log')
#plt.xlim(200,10**4)